## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os

from sklearn.metrics import silhouette_score
import numpy as np

import pandas as pd
import scanpy as sc
import multiscoresplot as msp

import matplotlib.pyplot as plt
import seaborn as sns

# Load data


In [ ]:
BASEPATH = f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling"

In [ ]:
adata_counts = sc.read_h5ad(
    os.path.join(BASEPATH, "UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad")
)

adata_counts

In [ ]:
adata_counts.obs["batch_number"] = pd.Categorical(
    adata_counts.obs["batch_number"].astype(int),
    categories=sorted(adata_counts.obs["batch_number"].astype(int).unique()),
    ordered=True,
)

In [ ]:
lineage_celltypes = [
    "Astrocytes",
    "qNSC1",
    "qNSC2",
    "ROB_qNSC2",
    "aNSC1",
    "aNSC2",
    "TAP",
    "NB",
    "Neurons",
]

# Integrate


In [ ]:
# Shared preprocessing base for all integration methods
# (normalize → log1p → scale → PCA on a copy of adata_counts)

adata_pp = adata_counts.copy()
adata_pp.raw = adata_pp.copy()  # store raw counts before any transformation

sc.pp.normalize_total(adata_pp, target_sum=1e4)
sc.pp.log1p(adata_pp)
sc.pp.scale(adata_pp, max_value=10)

sc.tl.pca(adata_pp, svd_solver="arpack", n_comps=50)
sc.pl.pca_variance_ratio(adata_pp, log=True, n_pcs=50)

print(adata_pp)

## 1. Harmony

**Harmony** iteratively adjusts the PCA embedding to remove batch effects. Fast, widely used, and integrates tightly with Scanpy. Produces `X_pca_harmony`.


In [ ]:
# sc.external.pp.harmony_integrate internally does Z_corr.T, but the Z_corr orientation
# changed across harmonypy versions, causing a shape mismatch in AnnData.
# We call harmonypy directly and normalise the shape ourselves.

import harmonypy as hm

adata_harmony = adata_pp.copy()
X_pca = adata_harmony.obsm["X_pca"].astype(np.float64)

ho = hm.run_harmony(X_pca, adata_harmony.obs, "batch_number")

Z = ho.Z_corr
# Ensure shape is (n_obs, n_components) regardless of harmonypy version
if Z.shape[0] != adata_harmony.n_obs:
    Z = Z.T
adata_harmony.obsm["X_pca_harmony"] = Z

sc.pp.neighbors(adata_harmony, use_rep="X_pca_harmony", n_neighbors=10, n_pcs=40)
sc.tl.umap(adata_harmony)
sc.tl.leiden(adata_harmony)

adata_harmony

## 2. Scanorama

Uses **mutual nearest neighbours across batches** in PCA space to produce a joint low-dimensional corrected embedding (`X_scanorama`). Handles situations where not all cell types are present in every batch.


In [ ]:
# Scanorama requires batches to be contiguous in the AnnData object.
# Sort cells by batch_number first, then run integration.

adata_scanorama = adata_pp[adata_pp.obs.sort_values("batch_number").index].copy()
sc.external.pp.scanorama_integrate(
    adata_scanorama,
    key="batch_number",
    basis="X_pca",
    adjusted_basis="X_scanorama",
)
sc.pp.neighbors(adata_scanorama, use_rep="X_scanorama", n_neighbors=10)
sc.tl.umap(adata_scanorama)
sc.tl.leiden(adata_scanorama)

adata_scanorama

## 3. BBKNN

**Batch-Balanced KNN** replaces the `sc.pp.neighbors` step with a batch-aware graph. Fast and often effective. Does not produce a corrected embedding — only the graph (and therefore UMAP) is batch-corrected.


In [ ]:
# BBKNN modifies the neighbor *graph* directly instead of producing a corrected embedding,
# so there is no separate "X_bbknn" representation — run PCA then replace the neighbors step.

adata_bbknn = adata_pp.copy()
sc.external.pp.bbknn(adata_bbknn, batch_key="batch_number", n_pcs=40)
sc.tl.umap(adata_bbknn)
sc.tl.leiden(adata_bbknn)

adata_bbknn

## 4. ComBat

Corrects batch effects at the **expression level** (before scaling/PCA). More aggressive than embedding-level methods — good when batches differ strongly in absolute expression levels.


In [ ]:
# ComBat must correct expression values *before* scaling/PCA,
# so we rebuild the preprocessing pipeline from scratch on a fresh copy.

adata_combat = adata_counts.copy()
adata_combat.raw = adata_combat  # store raw counts

sc.pp.normalize_total(adata_combat, target_sum=1e4)
sc.pp.log1p(adata_combat)
sc.pp.combat(adata_combat, key="batch_number")

sc.pp.scale(adata_combat, max_value=10)
sc.tl.pca(adata_combat, svd_solver="arpack", n_comps=50)
sc.pp.neighbors(adata_combat, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata_combat)
sc.tl.leiden(adata_combat)

adata_combat

# Dimensionality reduction


## Baseline (no integration)

PCA and UMAP on the preprocessed data **without** any batch correction. Use this as a visual reference against which each integration result can be judged.


In [ ]:
# Baseline: standard neighbors + UMAP on the preprocessed (but not batch-corrected) data.
# This shows the raw batch effects to judge how much each integration method improves things.

adata_base = adata_pp.copy()
sc.pp.neighbors(adata_base, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata_base)
sc.tl.leiden(adata_base)

adata_base

In [ ]:
sc.pl.pca(
    adata_base,
    color=[
        "batch_number",
        "celltype",
        "leiden",
        "dpt_pseudotime",
        "phase",
        "n_genes_by_counts",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_base,
    color=[
        "batch_number",
        "celltype",
        "leiden",
        "dpt_pseudotime",
        "phase",
        "n_genes_by_counts",
    ],
    wspace=0.4,
    ncols=3,
    title=[
        "Baseline — batch",
        "Baseline — celltype",
        "Baseline — leiden",
        "Baseline — pseudotime",
        "Baseline — phase",
        "Baseline — n_genes",
    ],
)

## 1. Harmony


In [ ]:
adata_harmony_lineage = adata_harmony[
    adata_harmony.obs["celltype"].isin(lineage_celltypes)
].copy()

In [ ]:
# Plot the Harmony-corrected embedding (X_pca_harmony)
sc.pl.embedding(
    adata_harmony,
    basis="pca_harmony",
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
# Plot UMAP
sc.pl.umap(
    adata_harmony,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_harmony_lineage,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_harmony_lineage,
    color=[
        "leiden",
        "name_old",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
# Integration quality metrics for Harmony.
# Average Silhouette Width (ASW) on the corrected embedding:
#   - ASW batch  → lower (closer to 0 or negative) = batches are well mixed
#   - ASW celltype → higher = biological cell-type structure is preserved
#
# Note: we subsample for speed; set n_sample = adata_harmony.shape[0] for exact results.

np.random.seed(42)
n_sample = min(5000, adata_harmony.shape[0])
idx_h = np.random.choice(adata_harmony.shape[0], n_sample, replace=False)

emb_h = adata_harmony.obsm["X_pca_harmony"][idx_h]
batch_h = adata_harmony.obs["batch_number"].astype(str).values[idx_h]
ct_h = adata_harmony.obs["celltype"].values[idx_h]

asw_batch_harmony = silhouette_score(emb_h, batch_h)
asw_ct_harmony = silhouette_score(emb_h, ct_h)

print(
    f"Harmony — ASW batch    (↓ = better batch mixing):       {asw_batch_harmony:.4f}"
)
print(f"Harmony — ASW celltype (↑ = better biology preserved):  {asw_ct_harmony:.4f}")

## 2. Scanorama


In [ ]:
adata_scanorama_lineage = adata_scanorama[
    adata_scanorama.obs["celltype"].isin(lineage_celltypes)
].copy()

In [ ]:
# Plot the Scanorama-corrected embedding (X_scanorama, first 2 components)
sc.pl.embedding(
    adata_scanorama,
    basis="scanorama",
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
# UMAP computed from X_scanorama — shows the batch-corrected structure
sc.pl.umap(
    adata_scanorama,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_scanorama_lineage,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
idx_s = np.random.choice(adata_scanorama.shape[0], n_sample, replace=False)

emb_s = adata_scanorama.obsm["X_scanorama"][idx_s]
batch_s = adata_scanorama.obs["batch_number"].astype(str).values[idx_s]
ct_s = adata_scanorama.obs["celltype"].values[idx_s]

asw_batch_scanorama = silhouette_score(emb_s, batch_s)
asw_ct_scanorama = silhouette_score(emb_s, ct_s)

print(
    f"Scanorama — ASW batch    (↓ = better batch mixing):       {asw_batch_scanorama:.4f}"
)
print(
    f"Scanorama — ASW celltype (↑ = better biology preserved):  {asw_ct_scanorama:.4f}"
)

## 3. BBKNN


In [ ]:
adata_bbknn_lineage = adata_bbknn[
    adata_bbknn.obs["celltype"].isin(lineage_celltypes)
].copy()

In [ ]:
# BBKNN operates entirely on the neighbour graph and does not produce a corrected
# low-dimensional embedding — there is no "integrated PCA" to plot here.
# Batch correction is only visible in the UMAP below (which is built from the BBKNN graph).
print("BBKNN: no corrected embedding available — see UMAP below.")

In [ ]:
# UMAP computed from the BBKNN batch-balanced graph
sc.pl.umap(
    adata_bbknn,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_bbknn_lineage,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
# For BBKNN, correction is in the graph; we use the UMAP embedding as proxy for ASW.
idx_b = np.random.choice(adata_bbknn.shape[0], n_sample, replace=False)

emb_b = adata_bbknn.obsm["X_umap"][idx_b]
batch_b = adata_bbknn.obs["batch_number"].astype(str).values[idx_b]
ct_b = adata_bbknn.obs["celltype"].values[idx_b]

asw_batch_bbknn = silhouette_score(emb_b, batch_b)
asw_ct_bbknn = silhouette_score(emb_b, ct_b)

print(f"BBKNN — ASW batch    (↓ = better batch mixing):       {asw_batch_bbknn:.4f}")
print(f"BBKNN — ASW celltype (↑ = better biology preserved):  {asw_ct_bbknn:.4f}")

## 4. ComBat


In [ ]:
adata_combat_lineage = adata_combat[
    adata_combat.obs["celltype"].isin(lineage_celltypes)
].copy()

In [ ]:
# PCA computed on ComBat-corrected expression — this IS already batch-corrected
sc.pl.pca(
    adata_combat,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_combat,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
sc.pl.umap(
    adata_combat_lineage,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

In [ ]:
idx_c = np.random.choice(adata_combat.shape[0], n_sample, replace=False)

emb_c = adata_combat.obsm["X_pca"][idx_c]
batch_c = adata_combat.obs["batch_number"].astype(str).values[idx_c]
ct_c = adata_combat.obs["celltype"].values[idx_c]

asw_batch_combat = silhouette_score(emb_c, batch_c)
asw_ct_combat = silhouette_score(emb_c, ct_c)

print(f"ComBat — ASW batch    (↓ = better batch mixing):       {asw_batch_combat:.4f}")
print(f"ComBat — ASW celltype (↑ = better biology preserved):  {asw_ct_combat:.4f}")

## Integration quality comparison

Side-by-side summary of the ASW metrics across all methods.

- **ASW batch** (x-axis left): lower / more negative = batches are better mixed (desired)
- **ASW celltype** (x-axis right): higher = biological structure is better preserved (desired)

Note: Harmony/Scanorama ASW is on their respective corrected embeddings; BBKNN on X_umap; ComBat on the corrected PCA. Methods are not strictly comparable in absolute terms, but the relative values within each metric are informative.


In [ ]:
results = pd.DataFrame(
    {
        "Method": ["Harmony", "Scanorama", "BBKNN*", "ComBat"],
        "ASW batch": [
            asw_batch_harmony,
            asw_batch_scanorama,
            asw_batch_bbknn,
            asw_batch_combat,
        ],
        "ASW celltype": [asw_ct_harmony, asw_ct_scanorama, asw_ct_bbknn, asw_ct_combat],
    }
).set_index("Method")

print(results.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colors_bar = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

axes[0].barh(results.index, results["ASW batch"], color=colors_bar)
axes[0].axvline(0, color="k", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("ASW batch  (↓ = better mixing)")
axes[0].set_title("Batch mixing")

axes[1].barh(results.index, results["ASW celltype"], color=colors_bar)
axes[1].axvline(0, color="k", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("ASW celltype  (↑ = better biology)")
axes[1].set_title("Biology preservation")

fig.suptitle("Integration quality (ASW scores)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# * BBKNN ASW is computed on X_umap (no corrected low-dim embedding is produced)

# Store data


In [ ]:
adata_harmony

In [ ]:
adata_harmony.write_h5ad(
    os.path.join(
        BASEPATH,
        "UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony.h5ad",
    )
)
adata_harmony_lineage.write_h5ad(
    os.path.join(
        BASEPATH,
        "UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony_lineage.h5ad",
    )
)

In [ ]:
adata_harmony_lineage.raw.X